# Kaggle analysis
Read verified real artifacts only. Pilot, tuning and final research roots stay separate. No placeholder figures or test-driven configuration selection.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
INPUT_DIR = Path(os.environ.get("SEQREC_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("SEQREC_WORK_DIR", "/kaggle/working"))
REPO = Path(os.environ.get("SEQREC_REPO_DIR", str(WORK_DIR / "sequential-recommendation")))
PYTHON = os.environ.get("SEQREC_PYTHON", str(WORK_DIR / "seqrec-env/bin/python"))
if not Path(PYTHON).exists():
    PYTHON = sys.executable
def run(*args):
    subprocess.run([PYTHON, *map(str, args)], cwd=REPO, check=True)
print({"repo": str(REPO), "python": PYTHON, "input": str(INPUT_DIR), "work": str(WORK_DIR)})


In [ ]:
session = json.loads((WORK_DIR / "seqrec_session.json").read_text())
REPO, PYTHON, BUNDLE = Path(session["repo"]), session["python"], Path(session["bundle"])
RESULTS_ROOT = REPO / "results/core"
run("scripts/verify_results.py", "--root", RESULTS_ROOT)
run("scripts/aggregate_results.py", "--root", RESULTS_ROOT, "--output", "results/tables/core")


In [ ]:
# Run these after the preselected E1 runs have completed.
RUN_E7_E8 = False
if RUN_E7_E8:
    for experiment in ["E7", "E8"]:
        run("scripts/run_analysis.py", "--experiment", experiment, "--bundle", BUNDLE,
            "--results-root", RESULTS_ROOT, "--output", "results/analysis/core", "--device", "cuda:0")


In [ ]:
import pandas as pd
table = REPO / "results/tables/core/per_seed_results.csv"
if table.exists():
    display(pd.read_csv(table))
else:
    print("No verified final results yet. Pilot and validation-only tuning artifacts are excluded.")


In [ ]:
EXPORT_PATH = WORK_DIR / "seqrec-analysis-session1.zip"
run("scripts/export_results.py", "--root", REPO / "results", "--output", EXPORT_PATH)
print(EXPORT_PATH)
